# 02: Hotel Data Analysis & Insights

**Author**: Nwaeke Calixtus, Esq  
**Date**: 2026-05-10  
**License**: MIT  

## Objective
Conduct statistical analysis on cleaned hotel data to extract business insights and identify trends.

---

## Contents
1. Load Cleaned Data
2. Booking Analysis
3. Revenue Analysis
4. Guest Segmentation
5. Temporal Patterns
6. Insights & Recommendations

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Plotting settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print('✓ Analysis environment ready')

## 1. Load Cleaned Data

In [ ]:
# Load cleaned data
from scripts.project_paths import data_file
cleaned_data_path = data_file('data','processed','cleaned_hotels.csv')

# If cleaned data doesn't exist, run cleaning first
if not cleaned_data_path.exists():
    print('Cleaned data not found. Running cleaning pipeline...')
    import sys
    sys.path.insert(0, '../scripts')
    from clean_data import clean
    
    raw_data = pd.read_csv(data_file('data','raw','hotels.csv'))
    df = clean(raw_data)
    df.to_csv(cleaned_data_path, index=False)
    print(f'✓ Cleaned data saved: {cleaned_data_path}')
else:
    df = pd.read_csv(cleaned_data_path)
    print(f'✓ Loaded cleaned data: {len(df):,} records')
    print(f'  Shape: {df.shape}')
    print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

## 2. Booking Analysis

In [ ]:
print('BOOKING STATISTICS')
print('='*80)

# Basic metrics
if 'booking_id' in df.columns:
    print(f'Total bookings: {df[\'booking_id\'].nunique():,}')

if 'hotel_id' in df.columns:
    print(f'Unique hotels: {df[\'hotel_id\'].nunique():,}')

if 'num_guests' in df.columns:
    print(f'\nGuest Statistics:')
    print(f'  Mean guests per booking: {df[\'num_guests\'].mean():.2f}')
    print(f'  Median: {df[\'num_guests\'].median():.0f}')
    print(f'  Max: {df[\'num_guests\'].max():.0f}')

print(f'\nData quality: {(df.notna().sum().sum() / (len(df)*df.shape[1])*100):.1f}% complete')

## 3. Revenue Analysis

In [ ]:
if 'room_rate' in df.columns:
    print('REVENUE ANALYSIS')
    print('='*80)
    
    # Price statistics
    print(f'Room Rate Statistics (USD):')
    print(f'  Mean: ${df[\'room_rate\'].mean():.2f}')
    print(f'  Median: ${df[\'room_rate\'].median():.2f}')
    print(f'  Std Dev: ${df[\'room_rate\'].std():.2f}')
    print(f'  Min: ${df[\'room_rate\'].min():.2f}')
    print(f'  Max: ${df[\'room_rate\'].max():.2f}')
    print(f'  Q1 (25%): ${df[\'room_rate\'].quantile(0.25):.2f}')
    print(f'  Q3 (75%): ${df[\'room_rate\'].quantile(0.75):.2f}')
    
    # Total revenue
    if 'total_price' in df.columns:
        total_revenue = df['total_price'].sum()
        print(f'\nTotal Revenue: ${total_revenue:,.2f}')
        print(f'Average Revenue per Booking: ${total_revenue/len(df):,.2f}')
    
    # Price distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df['room_rate'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Room Rate (USD)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Room Rates')
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot(df['room_rate'], vert=True)
    axes[1].set_ylabel('Room Rate (USD)')
    axes[1].set_title('Room Rate Box Plot')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. Guest Segmentation

In [ ]:
if 'num_guests' in df.columns:
    print('GUEST SEGMENTATION')
    print('='*80)
    
    guest_segments = df['num_guests'].value_counts().sort_index()
    print(f'\nBookings by Guest Count:')
    for guests, count in guest_segments.head(10).items():
        pct = (count / len(df)) * 100
        print(f'  {guests} guest(s): {count:7,} bookings ({pct:5.1f}%)')
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 5))
    guest_segments.head(10).plot(kind='bar', ax=ax, color='teal')
    ax.set_xlabel('Number of Guests')
    ax.set_ylabel('Number of Bookings')
    ax.set_title('Distribution of Bookings by Guest Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()

## 5. Temporal Patterns

In [ ]:
if 'check_in_date' in df.columns:
    # Parse dates if not already datetime
    if df['check_in_date'].dtype == 'object':
        df['check_in_date'] = pd.to_datetime(df['check_in_date'])
    
    print('TEMPORAL ANALYSIS')
    print('='*80)
    
    # Extract temporal features
    df['month'] = df['check_in_date'].dt.month
    df['day_of_week'] = df['check_in_date'].dt.day_name()
    
    # Monthly distribution
    monthly = df['month'].value_counts().sort_index()
    print(f'\nBookings by Month:')
    month_names = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for month, count in monthly.items():
        print(f'  {month_names[month]:>3}: {count:7,} bookings')
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    monthly.plot(kind='bar', ax=axes[0], color='coral')
    axes[0].set_xlabel('Month')
    axes[0].set_ylabel('Number of Bookings')
    axes[0].set_title('Bookings by Month')
    axes[0].set_xticklabels([month_names[i] for i in monthly.index], rotation=45)
    
    # Day of week
    dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_data = df['day_of_week'].value_counts().reindex(dow_order)
    dow_data.plot(kind='bar', ax=axes[1], color='lightgreen')
    axes[1].set_xlabel('Day of Week')
    axes[1].set_ylabel('Number of Bookings')
    axes[1].set_title('Bookings by Day of Week')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()

## 6. Key Insights & Recommendations

In [ ]:
print('\n' + '='*80)
print('KEY INSIGHTS & RECOMMENDATIONS')
print('='*80)

insights = f'''
DATA SUMMARY
  • {len(df):,} cleaned hotel booking records
  • {(df.notna().sum().sum() / (len(df)*df.shape[1])*100):.1f}% data completeness
  • Analysis ready for ML model development

BOOKING INSIGHTS
  • Data spans multiple properties and time periods
  • Mix of guest party sizes (1-{df['num_guests'].max():.0f} guests)
  • Diverse booking patterns and seasonal trends

REVENUE OPPORTUNITIES
  • Identify price elasticity for revenue optimization
  • Analyze seasonal demand for dynamic pricing
  • Segment guests for targeted marketing campaigns

NEXT STEPS
  1. ✓ Data exploration complete
  2. → Prepare data for predictive modeling
  3. → Develop occupancy/revenue forecasts
  4. → Optimize pricing strategies
  5. → Customer segmentation analysis

DOCUMENTATION
  • See docs/data_dictionary.md for field definitions
  • See docs/LEGAL.md for data governance & compliance
  • See CONTRIBUTING.md for development standards
'''

print(insights)